In [ ]:
import numpy as np
import polars as pl
from pathlib import Path
from loguru import logger

In [ ]:
INTER_DATA_PATH = "../../intermediary_data/"
print(f"Loading and saving data from/to {Path(INTER_DATA_PATH).resolve()}")


In [ ]:

vp_apache_path = Path(INTER_DATA_PATH) / Path("vital_periodic.csv")
vp_apache_path_parquet = Path(INTER_DATA_PATH) / Path("vital_periodic.parquet")
logger.info(f"Loading data from {vp_apache_path_parquet.resolve()}")

In [ ]:
df_vp_apache_valid = pl.read_parquet(vp_apache_path_parquet)
print(df_vp_apache_valid.head(n=1))

In [ ]:
df_vp_apache_valid.glimpse()

In [ ]:
columns = df_vp_apache_valid.columns
columns

In [ ]:
stats_vp_pl = df_vp_apache_valid.describe()

In [ ]:
print(stats_vp_pl)

In [ ]:
stats_pd = stats_vp_pl.to_pandas()
print(stats_pd)

# Percentiles Filtered Dataframe

In [ ]:
df_vp_apache_valid_lazy = pl.scan_parquet(vp_apache_path_parquet.resolve())

In [ ]:
df_vp_percentiles = pl.read_csv(source="vp_intermediary_data/vital_periodic_percentiles-2024-01-17.csv")

In [ ]:
df_vp_percentiles.head()

In [ ]:
df_vp_percentiles["temperature"][0]
df_vp_percentiles["temperature"][3]

In [ ]:
mask = pl.col("temperature").is_between(df_vp_percentiles["temperature"][0],df_vp_percentiles["temperature"][3] )
print(mask)

In [ ]:
df_vp_apache_valid_lazy.select("temperature").collect().describe()

In [ ]:
df_vp_percentiles.columns

In [ ]:
conditions_param_quantiles_1 = [pl.col(param).is_between(df_vp_percentiles[param][0],df_vp_percentiles[param][3]) for param in df_vp_percentiles.columns[1:]] # don't include index
print(conditions_param_quantiles_1)

In [ ]:
conditions_param_quantiles_2 = [pl.col(param).is_between(
    df_vp_percentiles.select(pl.col(param))[0], df_vp_percentiles.select(pl.col(param))[3]).not_()
    for param in df_vp_percentiles.columns[1:]] # don't include index
print(len(conditions_param_quantiles_2))



In [ ]:
# Todo: Wenn mit lazy eval nicht gescheid klappt mal im eager modus probiere
# Todo: https://docs.pola.rs/py-polars/html/reference/expressions/api/polars.when.html#polars.when

pl.when((df_vp_apache_valid_lazy.select(pl.col("temperature")) < df_vp_percentiles["temperature"][0]) | (df_vp_apache_valid_lazy.select(pl.col("temperature")) > df_vp_percentiles["temperature"][3])).then(np.nan).otherwise("temperature")

In [ ]:
import functools
import operator


#df_vp_apache_valid_lazy_filtered = df_vp_apache_valid_lazy.filter(
#df_vp_apache_valid_lazy_filtered = df_vp_apache_valid_lazy.filter(
#    #functools.reduce(operator.or_, conditions_param_quantiles)    
#).collect()

df_vp_apache_valid_lazy_filtered = df_vp_apache_valid_lazy.with_columns(
    pl.when(conditions_param_quantiles_2[0]).then(np.nan)
).collect()

In [ ]:
df_vp_apache_valid_lazy_filtered.describe()